# 11: O Consolidador (relatório final)

Toda análise termina no Consolidador. Antes dele, a resposta final era só a última mensagem do grafo:
às vezes o texto do setor, às vezes o diagnóstico, nunca tudo junto.

O Consolidador **não é um agente com tools**: é uma única chamada ao LLM com saída estruturada
(`RelatorioFinal`), seguida de ajustes em código e da formatação em markdown.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Uma análise completa da VALE3

Esta célula roda o pipeline inteiro (cerca de 30 segundos). A VALE3 costuma acionar o Investigator, então o
relatório terá diagnóstico.

In [2]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro import app

resultado = app.invoke({"messages": [HumanMessage("Analise a ação VALE3")]})
quem_falou = [m.name for m in resultado["messages"] if m.name]
print("Mensagens no histórico:", quem_falou)
print("loop_count:", resultado["loop_count"])

2026-11-08 17:49:16,478 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-08 17:49:17,785 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico de preços, balanço patrimonial e métricas principais da VALE3.'


2026-11-08 17:49:17,786 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-08 17:49:17,788 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


2026-11-08 17:49:18,620 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=VALE3.SA


2026-11-08 17:49:20,770 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ok (2434 chars)


... (38 outputs left out)

2026-11-08 17:49:46,019 - fluxo_financeiro - DEBUG - [router] FINISH → Consolidador


2026-11-08 17:49:46,020 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (9 msgs)


2026-11-08 17:49:48,394 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (2932 chars)


Mensagens no histórico: ['Supervisor', 'YFinance', 'Supervisor', 'Analyzer', 'Supervisor', 'Analyzer', 'Supervisor', 'Investigator', 'Consolidador']
loop_count: 5


## 2. O que o Consolidador recebe e devolve

Ele recebe só o pedido do usuário e as respostas de YFinance, Analyzer e Investigator (as tarefas do
Supervisor ficam de fora), cada uma dentro de `<resposta_agente>`. Devolve um `RelatorioFinal`:

In [3]:
from fluxo_financeiro.agents.consolidator import montar_insumos

print("\n\n".join(montar_insumos(resultado["messages"]))[:1500], "...")

<pedido_usuario>Analise a ação VALE3</pedido_usuario>

<resposta_agente nome="YFinance" n="1">
Dados para VALE3.SA:
Empresa: Vale S.A.
Preço Atual: R$ 64.04 (no dia: 2.19%)
Faixa de 52 semanas: R$ 59.42 a R$ 91.62
Valor de mercado: R$ 248.5 bi
P/L:  77.2x
ROE:  0.3%
DY:   6.82%
Setor: Materiais Básicos | Subsetor: Mineração de Minerais Metálicos
Maiores do setor na Brapi (Non-Energy Minerals, por valor de mercado): VALE3 é a 1ª de 12.
  1. VALE3 VALE S.A.: R$ 248.5 bi
  2. GGBR4 GERDAU S.A.: R$ 46.0 bi
  3. CMIN3 CSN MINERAÇÃO S.A.: R$ 28.3 bi
  4. KLBN4 KLABIN S.A.: R$ 21.7 bi
  5. GOAU4 METALURGICA GERDAU S.A.: R$ 14.0 bi
Sumario de Negócio:
A Vale S.A. (VALE3) é uma companhia brasileira de mineração e metais, com foco principal na produção de minério de ferro e de metais para transição energética. A empresa também atua na produção de pelotas de minério de ferro e, no segmento de metais básicos, em produtos como níquel, cobre, cobalto e metais do grupo da platina, além de subprodutos

In [4]:
relatorio = resultado["relatorio"]
for campo in ("resumo", "indicadores", "contexto", "riscos", "limitacoes"):
    print(f"--- {campo}:\n{getattr(relatorio, campo)}\n")

--- resumo:
A Vale S.A. (VALE3) encerrou o último pregão negociada a R$ 64.04, liderando o subsetor de mineração com valor de mercado de R$ 248.5 bi. A empresa apresentou queda de 53% no lucro líquido do trimestre (US$ 1.25 bi contra US$ 2.69 bi), motivada por fatores cíclicos como a retração nos preços globais do minério de ferro e elevação de custos. Os indicadores mostram um P/L de 77.2x, ROE de 0.3% e dividend yield de 6.82%, acompanhados por um Piotroski F-Score de 5/9 e Sharpe ratio de -0.11.

--- indicadores:
['Preço Atual: R$ 64.04', 'Faixa de 52 semanas: R$ 59.42 a R$ 91.62', 'Valor de mercado: R$ 248.5 bi', 'P/L: 77.2x', 'ROE: 0.3%', 'DY: 6.82%', 'Sharpe Ratio: -0.11', 'Sortino Ratio: -0.18', 'Max Drawdown: -31.5%', 'VaR 95% (diário): -2.91%', 'Beta (vs IBOV): 0.39', 'Alpha (anual): -3.4%', 'Piotroski F-Score: 5/9', 'SELIC (meta): 13.75% a.a.', 'IPCA acumulado em 12 meses: 4.58%']

--- contexto:
A Vale S.A. é uma mineradora global focada em minério de ferro e metais para tran

## 3. O diagnóstico é copiado, não resumido

O LLM é instruído a deixar `diagnostico` vazio; o código copia o texto do Investigator literalmente. Assim as
quatro seções (Causa, Natureza, Evidências, Impacto) nunca são reescritas.

In [5]:
texto_investigator = next((m.text for m in resultado["messages"] if m.name == "Investigator"), None)
print("Investigator rodou:", texto_investigator is not None)
print("Diagnóstico idêntico ao texto do Investigator:", relatorio.diagnostico == texto_investigator)

Investigator rodou: True
Diagnóstico idêntico ao texto do Investigator: True


## 4. A mesma função, chamada à parte

`consolidar(state)` é uma função comum: dá para chamá-la com qualquer state. Uma chamada ao LLM.

In [6]:
from fluxo_financeiro.agents.consolidator import consolidar, formatar_relatorio, relatorio_de_emergencia

de_novo = consolidar(resultado)
print("Indicadores na 2a chamada:", len(de_novo.indicadores), "| na análise:", len(relatorio.indicadores))

Indicadores na 2a chamada: 15 | na análise: 15


## 5. Do relatório estruturado ao markdown

`formatar_relatorio` monta o markdown em código. Seções vazias são omitidas.

In [7]:
from IPython.display import Markdown

Markdown(formatar_relatorio(relatorio, "VALE3"))

# Relatório: VALE3

## Resumo
A Vale S.A. (VALE3) encerrou o último pregão negociada a R$ 64.04, liderando o subsetor de mineração com valor de mercado de R$ 248.5 bi. A empresa apresentou queda de 53% no lucro líquido do trimestre (US$ 1.25 bi contra US$ 2.69 bi), motivada por fatores cíclicos como a retração nos preços globais do minério de ferro e elevação de custos. Os indicadores mostram um P/L de 77.2x, ROE de 0.3% e dividend yield de 6.82%, acompanhados por um Piotroski F-Score de 5/9 e Sharpe ratio de -0.11.

## Indicadores
- Preço Atual: R$ 64.04
- Faixa de 52 semanas: R$ 59.42 a R$ 91.62
- Valor de mercado: R$ 248.5 bi
- P/L: 77.2x
- ROE: 0.3%
- DY: 6.82%
- Sharpe Ratio: -0.11
- Sortino Ratio: -0.18
- Max Drawdown: -31.5%
- VaR 95% (diário): -2.91%
- Beta (vs IBOV): 0.39
- Alpha (anual): -3.4%
- Piotroski F-Score: 5/9
- SELIC (meta): 13.75% a.a.
- IPCA acumulado em 12 meses: 4.58%

## Notícias e setor
A Vale S.A. é uma mineradora global focada em minério de ferro e metais para transição energética, contando com infraestrutura logística própria. Notícias recentes mostram que o lucro líquido do terceiro trimestre de 2026 caiu 53% devido à desvalorização do minério de ferro globalmente, o que levou a uma postura mais defensiva de corretoras. No cenário setorial, concorrentes como a CSN buscam manobras de desalavancagem enquanto o mercado fomenta alternativas de financiamento para minerais críticos.

## Diagnóstico da anomalia
- **Causa identificada:** A queda de 53% no lucro líquido da Vale (VALE3) no 3T26 (US$ 1,25 bi versus US$ 2,69 bi no 3T25) foi motivada principalmente por fatores macroeconômicos e operacionais desfavoráveis: forte retração nos preços globais de referência do minério de ferro (pressionados pela fraqueza na demanda externa), elevação nos custos de produção e frete marítimo, e compressão das margens gerada pela menor receita realizada.
- **Natureza do evento:** Cíclico / Setorial (correlacionado aos ciclos de commodities e à desaceleração da demanda chinesa por minério de ferro, com pressões inflacionárias de insumos).
- **Evidências:** Comunicados de resultados e relatórios corporativos da Vale (relações com investidores), além de análises de mercado sobre preços de minério de ferro.
- **Impacto no investimento:** O movimento evidencia a alta sensibilidade do resultado da companhia a oscilações de preços de commodities. Não se trata de um evento contábil extraordinário ou *one-off* (como um impairment isolado), mas sim de um reflexo direto do ambiente de mercado adverso, exigindo atenção contínua à dinâmica de preços do minério e ao controle de custos operacionais.

## Riscos
- Sharpe Ratio de -0.11
- Sortino Ratio de -0.18
- Max Drawdown de -31.5%
- VaR 95% (diário) de -2.91%
- Alpha anual de -3.4%
- Queda de 53% no lucro líquido no 3T26
- Retração nos preços globais de referência do minério de ferro e elevação nos custos de produção e frete marítimo


## 6. Se o LLM do Consolidador falhar

A análise já custou várias chamadas; perder tudo no último passo seria ruim. Se `consolidar` levantar erro
(cota, rede, saída inválida), o nó usa `relatorio_de_emergencia`: as respostas dos agentes juntadas, sem LLM,
e `relatorio = None` no estado.

In [8]:
print(relatorio_de_emergencia(resultado["messages"])[:800], "...")

> Não foi possível consolidar o relatório; seguem as respostas dos agentes.

## YFinance
Dados para VALE3.SA:
Empresa: Vale S.A.
Preço Atual: R$ 64.04 (no dia: 2.19%)
Faixa de 52 semanas: R$ 59.42 a R$ 91.62
Valor de mercado: R$ 248.5 bi
P/L:  77.2x
ROE:  0.3%
DY:   6.82%
Setor: Materiais Básicos | Subsetor: Mineração de Minerais Metálicos
Maiores do setor na Brapi (Non-Energy Minerals, por valor de mercado): VALE3 é a 1ª de 12.
  1. VALE3 VALE S.A.: R$ 248.5 bi
  2. GGBR4 GERDAU S.A.: R$ 46.0 bi
  3. CMIN3 CSN MINERAÇÃO S.A.: R$ 28.3 bi
  4. KLBN4 KLABIN S.A.: R$ 21.7 bi
  5. GOAU4 METALURGICA GERDAU S.A.: R$ 14.0 bi
Sumario de Negócio:
A Vale S.A. (VALE3) é uma companhia brasileira de mineração e metais, com foco principal na produção de minério de ferro e de metais para transição energé ...


## Sem recomendação de compra ou venda

O relatório descreve o que os dados mostram e não dá veredito nem preço-alvo. Um "compre" escrito por um LLM a
partir de poucas fontes passaria uma confiança que os dados não sustentam.

## O que observar

- `limitacoes`: lista o que faltou ou falhou (busca sem resultado, etapa que não rodou). Parte delas é escrita em código, não depende de o LLM perceber.
- Os números de `indicadores` precisam aparecer nas respostas dos agentes: o prompt proíbe inventar ou recalcular.
- A segunda chamada a `consolidar` pode escrever frases diferentes da primeira (o LLM não é determinístico), mas os números são os mesmos, porque vêm dos insumos.
- O relatório de emergência é feio, mas não perde nada do que os agentes produziram.